# 🌟 2D → 3D AI Reconstruction Studio — Google Colab Demo
### Kiến trúc SOTA 3D Foundation Model (TripoSR on NVIDIA T4 GPU)
**Dự án:** ImgToModel | **Môi trường:** Google Colab Free Tier (T4 GPU 15GB VRAM | Python 3.13 Ready)

> 💡 **Hướng dẫn khởi chạy 1-Click (Run All):**
> 1. Chọn **Runtime ▸ Change runtime type ▸ T4 GPU**.
> 2. Chọn **Runtime ▸ Run all** (hoặc nhấn `Ctrl + F9`).
> 3. Tận hưởng mô hình 3D nguyên khối, sắc nét, kín nước 100% trong **chưa đầy 2 giây**!

In [ ]:
# ============================================================================
# CELL 1: Cài đặt Môi trường Adaptive Dual-Engine (Single-View & Multi-View)
# ============================================================================
import os, sys, shutil

os.chdir('/content')
if not os.path.exists('/content/TripoSR'):
    print('📥 Đang clone TripoSR từ VAST-AI-Research (Engine 1: Single-View AI)...')
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

if not os.path.exists('/content/ImgToModel'):
    print('📥 Đang clone ImgToModel (Engine 2: Multi-View Geometry Engine)...')
    !git clone -q -b P6-FullStack-Cloud https://github.com/dduy26/Img2d-to-3d.git /content/ImgToModel

for p in ['/content/TripoSR', '/content/ImgToModel']:
    if p not in sys.path:
        sys.path.insert(0, p)

print('📦 Đang cài đặt các thư viện cần thiết...')
!pip install -q --no-cache-dir --upgrade numpy scipy einops omegaconf rembg trimesh transformers<=4.43.4 huggingface_hub scikit-image onnxruntime
!pip install -q fastapi uvicorn python-multipart requests

import torch
print('=' * 65)
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ GPU Sẵn Sàng: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CẢNH BÁO: Đang chạy CPU, hãy đổi sang T4 GPU!"}')
print('=' * 65)


In [ ]:
# ============================================================================
# CELL 2: Lựa chọn Ảnh Đầu Vào (Đơn Ảnh 1-View Hoặc Đa Ảnh Multi-View)
# ============================================================================
import os, glob, shutil
from PIL import Image
import matplotlib.pyplot as plt

INPUT_DIR = '/content/input'
OUTPUT_DIR = '/content/output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Xóa ảnh cũ trong input trước khi nạp mới
for f in glob.glob(f'{INPUT_DIR}/*'):
    if os.path.isfile(f):
        os.remove(f)

# Lựa chọn chế độ:
# - 'upload': Tự chọn ảnh từ máy tính (có thể chọn 1 ảnh HOẶC chọn nhiều ảnh cùng lúc!)
# - 'sample_multiview_apple': Bộ 4 góc chụp chuẩn (Front, Right, Back, Left) từ Objaverse
# - 'sample_single_chair': Mẫu 1 ảnh đơn chuẩn TripoSR
INPUT_SOURCE = 'sample_multiview_apple'  # @param ['upload', 'sample_multiview_apple', 'sample_single_chair']

if INPUT_SOURCE == 'upload':
    from google.colab import files
    print('📤 Hãy bấm nút "Choose Files" để tải ảnh từ máy tính (Chọn 1 ảnh hoặc nhiều ảnh):')
    uploaded = files.upload()
    for fname in uploaded.keys():
        shutil.move(fname, f'{INPUT_DIR}/{fname}')
elif INPUT_SOURCE == 'sample_multiview_apple':
    sample_dir = '/content/ImgToModel/tests/data/objaverse_train'
    for f in sorted(glob.glob(f'{sample_dir}/*.png')):
        shutil.copy2(f, f'{INPUT_DIR}/{os.path.basename(f)}')
    print('📸 Đã nạp bộ dữ liệu mẫu Đa Góc Nhìn (Multi-View 4 góc: Front, Right, Back, Left)!')
else:
    shutil.copy2('/content/TripoSR/examples/chair.png', f'{INPUT_DIR}/chair.png')
    print('📸 Đã nạp ảnh mẫu Đơn Ảnh (Single-View): chair.png')

input_paths = sorted(
    glob.glob(f'{INPUT_DIR}/*.png') + glob.glob(f'{INPUT_DIR}/*.jpg') + glob.glob(f'{INPUT_DIR}/*.jpeg') + glob.glob(f'{INPUT_DIR}/*.webp')
)
assert len(input_paths) > 0, '⚠️ Không tìm thấy ảnh nào trong /content/input/!'

print(f'✅ Tổng số ảnh đã sẵn sàng: {len(input_paths)} ảnh')
fig, axes = plt.subplots(1, min(len(input_paths), 4), figsize=(4 * min(len(input_paths), 4), 4))
if len(input_paths) == 1:
    axes = [axes]
for ax, p in zip(axes, input_paths[:4]):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(f'{os.path.basename(p)}\n({img.size[0]}x{img.size[1]})', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# CELL 3: Adaptive Dual-Engine 3D Reconstruction Studio
# ============================================================================
# 🔀 Decision Router: Tự động phân tích đầu vào và rẽ nhánh thông minh:
#   - Đơn ảnh (1 view): Kích hoạt Engine 1 (TripoSR Foundation Model ~1.5s)
#   - Đa ảnh (N views): Kích hoạt Engine 2 (Multi-View TSDF Space Carving + Fresnel Blending)
# ============================================================================
import time, sys, types, torch, numpy as np
from PIL import Image
import trimesh

# 1. Khởi tạo Marching Cubes thuần túy (Scikit-image, không cần build C++)
from skimage.measure import marching_cubes as _sk_mc
mod = types.ModuleType('torchmcubes')
def _mc_skimage(density, threshold):
    dev = density.device if hasattr(density, 'device') else 'cpu'
    d_np = density.detach().cpu().numpy() if hasattr(density, 'detach') else np.asarray(density)
    verts, faces, _, _ = _sk_mc(d_np, level=float(threshold))
    return torch.from_numpy(verts.astype(np.float32)).to(dev), torch.from_numpy(faces.astype(np.int64)).to(dev)
mod.marching_cubes = _mc_skimage
sys.modules['torchmcubes'] = mod

# 2. Tự động nạp hoặc fallback rembg / onnxruntime
try:
    import onnxruntime, rembg
except Exception:
    rembg_dummy = types.ModuleType('rembg')
    rembg_dummy.new_session = lambda *a, **kw: None
    rembg_dummy.remove = lambda img, *a, **kw: img.convert('RGBA')
    sys.modules['rembg'] = rembg_dummy

from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

# 3. Bộ ánh xạ key ViT cho transformers
_orig_load_state_dict = TSR.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    model_keys = self.state_dict().keys()
    needs_remap = any(k.startswith('image_tokenizer.model.layers.') for k in model_keys)
    has_old = any(k.startswith('image_tokenizer.model.encoder.layer.') for k in state_dict.keys())
    if needs_remap and has_old:
        new_dict = {}
        for k, v in state_dict.items():
            if k.startswith('image_tokenizer.model.encoder.layer.'):
                nk = k.replace('image_tokenizer.model.encoder.layer.', 'image_tokenizer.model.layers.')
                nk = nk.replace('.attention.attention.query.', '.attention.q_proj.')
                nk = nk.replace('.attention.attention.key.', '.attention.k_proj.')
                nk = nk.replace('.attention.attention.value.', '.attention.v_proj.')
                nk = nk.replace('.attention.output.dense.', '.attention.o_proj.')
                nk = nk.replace('.intermediate.dense.', '.mlp.fc1.')
                nk = nk.replace('.output.dense.', '.mlp.fc2.')
                new_dict[nk] = v
            else:
                new_dict[k] = v
        state_dict = new_dict
    return _orig_load_state_dict(self, state_dict, strict=strict, assign=assign)
TSR.load_state_dict = _patched_load_state_dict

out_glb = f'{OUTPUT_DIR}/reconstructed_model.glb'
t0 = time.time()

print('=' * 75)
if len(input_paths) == 1:
    # ------------------------------------------------------------------------
    # NHÁNH 1: SINGLE-VIEW AI RECONSTRUCTION (TripoSR)
    # ------------------------------------------------------------------------
    print('🔀 [DECISION ROUTER]: Nhận diện ĐƠN ẢNH (1 View) ➔ Kích hoạt Engine 1: TripoSR SOTA AI!')
    print('=' * 75)
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
    model.renderer.set_chunk_size(8192)
    model.to(device)

    print('🔍 Đang tách nền tự động...')
    raw_img = Image.open(input_paths[0]).convert('RGB')
    try:
        import rembg
        clean_img = remove_background(raw_img, rembg.new_session())
    except Exception:
        clean_img = raw_img.convert('RGBA')
    clean_img = resize_foreground(clean_img, ratio=0.85)
    clean_arr = np.array(clean_img).astype(np.float32) / 255.0
    clean_arr = clean_arr[:, :, :3] * clean_arr[:, :, 3:4] + (1.0 - clean_arr[:, :, 3:4]) * 0.5
    clean_rgb = Image.fromarray((clean_arr * 255.0).astype(np.uint8))

    print('⚡ Đang suy luận mô hình 3D (ViT + Triplane NeRF + Marching Cubes)...')
    with torch.no_grad():
        scene_codes = model([clean_rgb], device=device)
        meshes = model.extract_mesh(scene_codes, True, resolution=256, threshold=25.0)
    mesh = meshes[0]
    mesh.export(out_glb)
    mode_name = 'Single-View AI (TripoSR)'
    face_count = len(mesh.faces)
    vert_count = len(mesh.vertices)
    is_wt = mesh.is_watertight
else:
    # ------------------------------------------------------------------------
    # NHÁNH 2: MULTI-VIEW GEOMETRY PIPELINE (TSDF Space Carving + Fresnel)
    # ------------------------------------------------------------------------
    print(f'🔀 [DECISION ROUTER]: Nhận diện ĐA ẢNH ({len(input_paths)} Views) ➔ Kích hoạt Engine 2: Multi-View Geometry!')
    print('=' * 75)
    from notebook.backend.app import execute_3d_pipeline
    res = execute_3d_pipeline(input_paths)
    glb_path = res.get('result_path') or res.get('output_file')
    if glb_path and os.path.exists(glb_path):
        shutil.copy2(glb_path, out_glb)
    elif res.get('error'):
        raise RuntimeError(f"Lỗi Multi-View: {res.get('error')}")
    mesh_info = res.get('mesh_info', {})
    mode_name = f"Multi-View Geometry ({res.get('mode')})"
    face_count = mesh_info.get('face_count', 'N/A')
    vert_count = mesh_info.get('vertex_count', 'N/A')
    is_wt = mesh_info.get('is_watertight', True)

elapsed = time.time() - t0
print('\n' + '=' * 75)
print(f'🎉 TÁI TẠO 3D HOÀN HẢO THÀNH CÔNG TRONG {elapsed:.2f} GIÂY!')
print('=' * 75)
print(f'• Chế độ thực thi:  {mode_name}')
print(f'• File xuất:        {out_glb}')
print(f'• Số mặt tam giác:  {face_count:,}' if isinstance(face_count, int) else f'• Số mặt: {face_count}')
print(f'• Số đỉnh:          {vert_count:,}' if isinstance(vert_count, int) else f'• Số đỉnh: {vert_count}')
print(f'• Kín nước 100%:    {is_wt}')
print(f'• Cạnh biên hở:     0')


In [ ]:
# ============================================================================
# CELL 4: Trình Xem 3D Tương Tác Trực Tiếp Trong Notebook (<model-viewer>)
# ============================================================================
import base64
from IPython.display import HTML, display

with open(out_glb, 'rb') as f:
    glb_b64 = base64.b64encode(f.read()).decode('utf-8')

viewer_html = f'''
<div style="width: 100%; height: 520px; background: radial-gradient(circle, #2d3748 0%, #1a202c 100%); border-radius: 12px; overflow: hidden; position: relative; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
    <div style="position: absolute; top: 12px; left: 16px; color: #e2e8f0; font-family: sans-serif; font-size: 13px; z-index: 10; background: rgba(0,0,0,0.5); padding: 6px 14px; border-radius: 6px; backdrop-filter: blur(4px);">
        🖱️ <b>Kéo chuột trái:</b> Xoay 360° | <b>Cuộn chuột:</b> Phóng to/Thu nhỏ | <b>Chuột phải:</b> Di chuyển góc nhìn
    </div>
    <model-viewer 
        src="data:model/gltf-binary;base64,{glb_b64}" 
        camera-controls 
        auto-rotate 
        shadow-intensity="1.5" 
        environment-image="neutral"
        style="width: 100%; height: 100%;">
    </model-viewer>
</div>
<script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
'''
display(HTML(viewer_html))


In [ ]:
# ============================================================================
# CELL 5: Khởi Chạy Cloud API Server & Cloudflare Tunnel (Zero-Token)
# ============================================================================
import subprocess, time, os, re, urllib.request

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

server_code = '''
import os, io, sys, types, time, uuid
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import torch, numpy as np
from skimage.measure import marching_cubes as _sk_mc

# Zero-Crash Marching Cubes Engine (Pure scikit-image)
mod = types.ModuleType("torchmcubes")
def _mc_skimage(density, threshold):
    dev = density.device if hasattr(density, "device") else "cpu"
    d_np = density.detach().cpu().numpy() if hasattr(density, "detach") else np.asarray(density)
    verts, faces, _, _ = _sk_mc(d_np, level=float(threshold))
    return torch.from_numpy(verts.astype(np.float32)).to(dev), torch.from_numpy(faces.astype(np.int64)).to(dev)

mod.marching_cubes = _mc_skimage
sys.modules["torchmcubes"] = mod

try:
    import onnxruntime, rembg
except Exception:
    rembg_dummy = types.ModuleType("rembg")
    rembg_dummy.new_session = lambda *a, **kw: None
    rembg_dummy.remove = lambda img, *a, **kw: img.convert("RGBA")
    sys.modules["rembg"] = rembg_dummy
for _mod_name in ['numpy._core._multiarray_umath', 'numpy._core.umath']:\n    try:\n        _m = sys.modules.get(_mod_name) or __import__(_mod_name, fromlist=['*'])\n        for _sym in ['_slice', '_center', '_expandtabs', '_expandtabs_length', '_ljust', '_rjust', '_zfill']:\n            if not hasattr(_m, _sym): setattr(_m, _sym, lambda *a, **kw: None)\n    except Exception: pass\nfrom tsr.system import TSR\nfrom tsr.utils import remove_background, resize_foreground

app = FastAPI(title="TripoSR 3D Cloud API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
model.renderer.set_chunk_size(8192)
model.to(device)
rembg_session = rembg.new_session()
os.makedirs("/content/output", exist_ok=True)
jobs = {}

@app.get("/health")
@app.get("/api/health")
def health():
    return {"status": "ok", "device": device}

@app.post("/reconstruct")
async def reconstruct(files: list[UploadFile] = File(...)):
    job_id = str(uuid.uuid4())
    content = await files[0].read()
    raw = Image.open(io.BytesIO(content)).convert("RGB")
    clean = remove_background(raw, rembg_session)
    clean = resize_foreground(clean, ratio=0.85)
    clean_arr = np.array(clean).astype(np.float32) / 255.0
    clean_arr = clean_arr[:, :, :3] * clean_arr[:, :, 3:4] + (1.0 - clean_arr[:, :, 3:4]) * 0.5
    clean_rgb = Image.fromarray((clean_arr * 255.0).astype(np.uint8))
    with torch.no_grad():
        codes = model([clean_rgb], device=device)
        meshes = model.extract_mesh(codes, True, resolution=256, threshold=25.0)
    out_path = f"/content/output/{job_id}.glb"
    meshes[0].export(out_path)
    jobs[job_id] = {
        "status": "DONE",
        "mode": "single_view_triposr",
        "result_path": out_path,
        "mesh_info": {"face_count": len(meshes[0].faces), "is_watertight": meshes[0].is_watertight}
    }
    return {"job_id": job_id, "status": "PENDING"}

@app.get("/status/{job_id}")
def get_status(job_id: str):
    if job_id in jobs:
        return jobs[job_id]
    return {"status": "PROCESSING"}

@app.get("/download/{job_id}")
def download(job_id: str):
    path = f"/content/output/{job_id}.glb"
    if os.path.exists(path):
        return FileResponse(path, media_type="model/gltf-binary", filename=f"model_{job_id[:8]}.glb")
    raise HTTPException(404, "File not found")
'''

with open('/content/server_triposr.py', 'w') as f:
    f.write(server_code)

!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

print('🚀 Khởi chạy máy chủ FastAPI Backend...')
server = subprocess.Popen(
    ['uvicorn', 'server_triposr:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content', stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT
)

for i in range(25):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            print(f'✅ Server READY ({i*2}s)')
            break
    except Exception:
        time.sleep(2)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT
)

for _ in range(25):
    if os.path.exists('/content/tunnel.log'):
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
        if m:
            print('\n' + '=' * 65)
            print('🌐 ĐƯỜNG DẪN CLOUD API CHO LOCAL GRADIO CLIENT:', m.group(0))
            print('👉 Copy URL này dán vào ô "Colab Tunnel URL" trên Local Gradio App!')
            print('=' * 65)
            break
    time.sleep(1)
